# Recommendations

Builds a candidate track ranking for a target playlist using model coeffficients and the available track/artist metadata. see `data/processed/recommendations.csv`

In [10]:
import os
import numpy as np
import pandas as pd

In [11]:
pf_path="data/processed/playlist_features.csv"
coeff_path="data/processed/growth_model_coeffficients.csv"

pf=pd.read_csv(pf_path)
coeff_df=pd.read_csv(coeff_path) if os.path.exists(coeff_path) else pd.DataFrame(columns=["feature","coeffficient"])

pf=pf.set_index("playlist_id", drop=False)

# helper; need to return as series
def get_numeric_col(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    else:
        return pd.Series(0.0, index=df.index)

coeff=dict(zip(coeff_df["feature"].astype(str),  pd.to_numeric(coeff_df["coeffficient"], errors="coerce").fillna(0.0)))

print("playlist_features:", pf.shape)
print("coeff rows:", len(coeff_df))


playlist_features: (1, 52)
coeff rows: 0


In [12]:
tracks_path = "data/processed/tracks.csv"
artists_path = "data/processed/artists.csv"
track_feat_path = "data/processed/track_features.csv"

if os.path.exists(tracks_path) and os.path.exists(artists_path):
    tracks_df = pd.read_csv(tracks_path)
    artists_df = pd.read_csv(artists_path)
    track_features_df = (
        pd.read_csv(track_feat_path)
        if os.path.exists(track_feat_path)
        else tracks_df.copy()
    )
else:
    tracks_df = pf.reset_index(drop=True).copy()
    tracks_df["track_id"] = tracks_df["playlist_id"]
    tracks_df["artist_id"] = tracks_df["playlist_id"]
    tracks_df["track_name"] = tracks_df.get("playlist_name", "unknown")

    artists_df = tracks_df[["artist_id"]].copy()
    artists_df["artist_name"] = tracks_df.get("playlist_name", "unknown")
    artists_df["genres"] = ""

    track_features_df = tracks_df.copy()

for col in ["track_id","artist_id"]:
    if col not in track_features_df.columns:
        raise ValueError(f"track_features_df must include {col}.")

print("tracks_df:", tracks_df.shape)
print("artists_df:", artists_df.shape)
print("track_features_df:", track_features_df.shape)

tracks_df: (1, 55)
artists_df: (1, 3)
track_features_df: (1, 55)


In [13]:
target_playlist_id = pf["playlist_id"].iloc[0]
if target_playlist_id not in pf.index:
    raise ValueError("target_playlist_id not found in playlist_features.")

playlist_genre_cols = [c for c in pf.columns if c.startswith("genre_")]
playlist_genre_base = playlist_genre_cols.copy()

w_pop = float(coeff.get("track_popularity_mean", coeff.get("track_popularity", 0.0)))
w_artist_pop = float(coeff.get("artist_popularity_mean", coeff.get("artist_popularity", 0.0)))
w_log_af = float(coeff.get("log_artist_followers_mean", coeff.get("log_artist_followers", 0.0)))
w_age = float(coeff.get("track_age_days_mean", coeff.get("track_age_days", 0.0)))
w_explicit = float(coeff.get("explicit_mean", coeff.get("explicit", 0.0)))

genre_weights = np.array([float(coeff.get(c + "_mean", coeff.get(c, 0.0))) for c in playlist_genre_base], dtype=float)

# target_playlist_id


In [14]:
def _to_genre_set(x):
    if pd.isna(x):
        return set()
    if isinstance(x, (list, tuple, set)):
        return set([str(g).strip().lower() for g in x if str(g).strip()])
    s = str(x).strip()
    s = s.strip("[]")
    parts = [p.strip(" '\"").lower() for p in s.replace(";", ",").split(",")]
    return set([p for p in parts if p])

genre_col_name = None
for cand in ["genres","artist_genres","genres_list"]:
    if cand in artists_df.columns:
        genre_col_name = cand
        break
if genre_col_name is None:
    raise ValueError("artists_df must include a genres column (e.g., 'genres').")

artist_genre_map = dict(zip(artists_df["artist_id"].astype(str), artists_df[genre_col_name].apply(_to_genre_set)))

track_features_df = track_features_df.copy()
track_features_df["artist_id"] = track_features_df["artist_id"].astype(str)

In [15]:
target_genre_vec = pf.loc[target_playlist_id, playlist_genre_base].to_numpy(dtype=float) if len(playlist_genre_base) else np.array([], dtype=float)

cand_df = track_features_df.copy()

for g_base in playlist_genre_base:
    g_name = g_base.replace("genre_", "").replace("_mean", "").strip().lower()
    cand_df[g_base] = cand_df["artist_id"].map(lambda aid: 1.0 if g_name in artist_genre_map.get(aid, set()) else 0.0)

if "track_popularity" not in cand_df.columns and "popularity" in cand_df.columns:
    cand_df["track_popularity"] = cand_df["popularity"]

if "artist_popularity" not in cand_df.columns:
    if "popularity" in artists_df.columns:
        cand_df = cand_df.merge(artists_df[["artist_id","popularity"]].rename(columns={"popularity":"artist_popularity"}), on="artist_id", how="left")

if "explicit" not in cand_df.columns and "explicit" in tracks_df.columns:
    cand_df = cand_df.merge(tracks_df[["track_id","artist_id","explicit"]], on=["track_id","artist_id"], how="left")

if "track_age_days" not in cand_df.columns:
    date_cols = [c for c in tracks_df.columns if "release" in c.lower() and "date" in c.lower()]
    if date_cols:
        rd = pd.to_datetime(tracks_df[date_cols[0]], errors="coerce")
        ref = rd.max()
        tmp = tracks_df[["track_id","artist_id"]].copy()
        tmp["track_age_days"] = (ref - rd).dt.days
        cand_df = cand_df.merge(tmp, on=["track_id","artist_id"], how="left")


In [16]:
track_genre_mat = cand_df[playlist_genre_base].to_numpy(dtype=float) if len(playlist_genre_base) else np.zeros((len(cand_df),0), dtype=float)

cand_df["score_pop"] = w_pop * get_numeric_col(cand_df, "track_popularity")
cand_df["score_artist_pop"] = w_artist_pop * get_numeric_col(cand_df, "artist_popularity")
cand_df["score_log_artist_followers"] = w_log_af * get_numeric_col(cand_df, "log_artist_followers")
cand_df["score_age"] = w_age * get_numeric_col(cand_df, "track_age_days")
cand_df["score_explicit"] = w_explicit * get_numeric_col(cand_df, "explicit")

cand_df["genre_match"] = (track_genre_mat @ target_genre_vec) if len(playlist_genre_base) else 0.0
cand_df["score_genre"] = (track_genre_mat @ genre_weights) if len(playlist_genre_base) else 0.0

cand_df["score_total"] = (
    cand_df["score_pop"]
    + cand_df["score_artist_pop"]
    + cand_df["score_log_artist_followers"]
    + cand_df["score_age"]
    + cand_df["score_explicit"]
    + 0.1 * cand_df["genre_match"]
    + 1.0*cand_df["score_genre"]
)

In [17]:
meta_cols = [c for c in ["track_id","track_name","artist_id","artist_name"] if c in tracks_df.columns]
if meta_cols:
    meta = tracks_df[meta_cols].copy()
    if "artist_name" not in meta.columns and "name" in artists_df.columns:
        meta = meta.merge(artists_df[["artist_id","name"]].rename(columns={"name":"artist_name"}), on="artist_id", how="left")
    cand_df = cand_df.merge(meta.drop_duplicates(subset=["track_id","artist_id"]), on=["track_id","artist_id"], how="left")

out_cols=[c for c in ["track_id","track_name","artist_name","score_total","genre_match","track_popularity","artist_popularity","log_artist_followers","track_age_days","explicit"] if c in cand_df.columns]
rec_df=cand_df.sort_values("score_total", ascending=False).head(50)[out_cols].copy()
# rec_df.drop(columns=["track_id"], errors="ignore").head()

In [18]:
os.makedirs("data/processed", exist_ok=True)
out_path="data/processed/recommendations.csv"
rec_df.to_csv(out_path, index=False)
out_path

'data/processed/recommendations.csv'